**FinBERT + BiLSTM + Attention (modèle final)**

**Différence clé vs modele 02 :** `return_sequences=True` sur le BiLSTM + couche d'attention apprise qui pondère les 128 tokens.

***Architecture :***
```
Tweet → FinBERT → [128 × 768] → BiLSTM(256, return_seq=True) → [128 × 512]
     → AttentionLayer → context [512] + weights [128]
     → Dropout → Dense(128) → Dense(3)
```

**Import & Configuration**

In [1]:
import os, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score
)

# ── Seeds ────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# ── Chemins ──────────────────────────────────────────────────────────────────
DATA_DIR  = Path('data/processed')
RES_DIR   = Path('results')
MODEL_DIR = Path('models/fintwit_bilstm_attention')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparamètres ──────────────────────────────────────────────────────────
BACKBONE     = 'ProsusAI/finbert'   # même tokenizer que le preprocessing (attention on a ajouter des tokens speciaux)
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)

special_tokens = [f'TICKER_{s}' for s in [
    'AAPL','MSFT','GOOGL','AMZN','TSLA','META','NVDA','BRK','JPM',
    'JNJ','V','PG','UNH','HD','MA','PYPL','BAC','DIS','ADBE','NFLX',
    'CMCSA','XOM','VZ','INTC','T','CSCO','PFE'
]] + ['AMOUNT_']

tokenizer.add_tokens(special_tokens)
MAX_LENGTH   = 128
BATCH_SIZE   = 16
GRAD_ACCUM   = 2
LR_BERT      = 2e-5
LR_HEAD      = 1e-3
WEIGHT_DECAY = 0.01
MAX_EPOCHS   = 10
PATIENCE     = 3
WARMUP_RATIO = 0.10
FREEZE_LAYERS = 8

LABEL_NAMES = ['négatif', 'neutre', 'positif']
print('Configuration OK')

C:\Users\cherb\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu
Configuration OK


**Chargment des données**

In [2]:
train_enc = torch.load(DATA_DIR / 'train_encodings.pt')
val_enc   = torch.load(DATA_DIR / 'val_encodings.pt')
test_enc  = torch.load(DATA_DIR / 'test_encodings.pt')

print(f'Train  : {train_enc["input_ids"].shape[0]} tweets')
print(f'Val    : {val_enc["input_ids"].shape[0]} tweets')
print(f'Test   : {test_enc["input_ids"].shape[0]} tweets')

C:\Users\cherb\AppData\Local\Temp\ipykernel_32904\1672103546.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_enc = torch.load(DATA_DIR / 'train_encodings.pt')


Train  : 29576 tweets
Val    : 6338 tweets
Test   : 6338 tweets


C:\Users\cherb\AppData\Local\Temp\ipykernel_32904\1672103546.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  val_enc   = torch.load(DATA_DIR / 'val_encodings.pt')
C:\Use

In [3]:
class TweetDataset(Dataset):
    def __init__(self, encodings):
        self.input_ids      = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels         = encodings['labels']

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels':         self.labels[idx]
        }

train_loader = DataLoader(TweetDataset(train_enc), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(TweetDataset(val_enc),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(TweetDataset(test_enc),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

Batches — Train: 1849 | Val: 397 | Test: 397


**Score de performance précedents**

In [5]:
with open(RES_DIR / 'metrics_baseline.json') as f:
    baseline = json.load(f)
with open(RES_DIR / 'metrics_01.json') as f:
    metrics_01 = json.load(f)
with open(RES_DIR / 'metrics_02.json') as f:
    metrics_02 = json.load(f)

print('=== RÉFÉRENCES À BATTRE ===')
print(f'  Baseline TF-IDF    F1-macro : {baseline["f1_macro"]:.4f}')
print(f'  01 FinBERT seul   F1-macro : {metrics_01["f1_macro"]:.4f}')
print(f'  02 + BiLSTM       F1-macro : {metrics_02["f1_macro"]:.4f}  ← plancher direct')

FileNotFoundError: [Errno 2] No such file or directory: 'results\\metrics_01.json'

**Class weights**

In [ ]:
with open(RES_DIR / 'class_weights.json') as f:
    cw_data = json.load(f)

cw = cw_data['weights']
class_weights_tensor = torch.tensor(
    [cw['0'], cw['1'], cw['2']], dtype=torch.float
).to(DEVICE)
print('Class weights :', {LABEL_NAMES[i]: round(cw[str(i)], 3) for i in range(3)})

**Couche d'Attention**

**Formulation mathématique (Bahdanau et al., 2015) :**

Pour chaque pas de temps $t$ de la sortie BiLSTM $h_t \in \mathbb{R}^{512}$ :

$$e_t = W_2 \cdot \tanh(W_1 \cdot h_t)$$

$$\alpha_t = \frac{\exp(e_t)}{\sum_{k=1}^{T} \exp(e_k)} \quad \text{(softmax)}$$

$$\text{context} = \sum_{t=1}^{T} \alpha_t \cdot h_t$$

- $\alpha_t$ : poids d'attention du token $t$ (entre 0 et 1, somme = 1)
- context : vecteur résumé pondéré par l'importance de chaque token

**Pourquoi cette attention est interprétable ?**  
Un seul scalaire $\alpha_t$ par token → facile à visualiser.  
vs BERT : 144 têtes d'attention (12 couches × 12 têtes) → impossible à agréger simplement.

**Nuance importante (pour la soutenance) :**  
Jain & Wallace (2019) montrent que l'attention n'est pas toujours une explication causale.  
Wiegreffe & Pinter (2019) répondent qu'elle reste informative dans la majorité des cas.  
→ On l'utilise comme indicateur d'importance, pas comme preuve de causalité.

In [ ]:
class AttentionLayer(nn.Module):
    """
    Couche d'attention additive (Bahdanau style).

    Entrée  : séquence BiLSTM  (batch, seq_len, hidden_dim)  — hidden_dim = 512
    Sorties :
        context_vector  (batch, hidden_dim)  — résumé pondéré
        attention_weights (batch, seq_len)   — poids par token (exploitables pour XAI)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # W1 projette chaque hidden state dans un espace intermédiaire
        self.W1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        # W2 projette vers un scalaire de score
        self.W2 = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_output, attention_mask=None):
        """
        lstm_output    : (batch, seq_len, 512)
        attention_mask : (batch, seq_len) — 1=token réel, 0=padding
        """
        # Score pour chaque token : e_t = W2 · tanh(W1 · h_t)
        scores = self.W2(torch.tanh(self.W1(lstm_output)))  # (batch, seq_len, 1)
        scores = scores.squeeze(-1)                          # (batch, seq_len)

        # Masquer les tokens de padding avant le softmax
        # (on ne veut pas que le padding influence les poids d'attention)
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float('-inf'))

        # Softmax → poids normalisés entre 0 et 1
        alpha = F.softmax(scores, dim=-1)  # (batch, seq_len)

        # Vecteur contexte = somme pondérée des hidden states
        context = torch.bmm(alpha.unsqueeze(1), lstm_output)  # (batch, 1, 512)
        context = context.squeeze(1)                           # (batch, 512)

        return context, alpha


print('AttentionLayer définie.')

**Architecture complète**

**Différence clé vs 02 :**  
Le BiLSTM retourne toute la séquence (`return_sequences=True` équivalent en PyTorch).  
L'AttentionLayer apprend à pondérer chaque token → le context vector est plus informatif  
que le simple dernier pas de temps utilisé dans le modèle 2

In [ ]:
class FinBertBiLSTMAttention(nn.Module):
    """
    FinBERT + BiLSTM + Attention + Classification Head.

    Flux :
      input_ids, attention_mask
        → FinBERT → hidden states [batch, 128, 768]
        → BiLSTM(256, bidirectionnel, return_sequences=True)
           → séquence complète [batch, 128, 512]
        → AttentionLayer
           → context vector [batch, 512]
           → attention weights [batch, 128]  ← exploitables pour XAI
        → Dropout(0.3)
        → Linear(512 → 128) + ReLU
        → Dropout(0.2)
        → Linear(128 → 3)
    """
    def __init__(self, backbone_name, num_classes=3, lstm_hidden=256,
                 dropout1=0.3, dropout2=0.2, freeze_layers=8):
        super().__init__()

        self.bert = AutoModel.from_pretrained(backbone_name)
        hidden_size = self.bert.config.hidden_size  # 768

        self._freeze_bert_layers(freeze_layers)

        # BiLSTM — on garde TOUTE la séquence (pas seulement le dernier pas)
        self.bilstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Couche d'attention sur la séquence BiLSTM complète
        self.attention = AttentionLayer(lstm_hidden * 2)  # 512

        # Tête de classification
        self.drop1      = nn.Dropout(dropout1)
        self.dense      = nn.Linear(lstm_hidden * 2, 128)
        self.relu       = nn.ReLU()
        self.drop2      = nn.Dropout(dropout2)
        self.classifier = nn.Linear(128, num_classes)

    def _freeze_bert_layers(self, n_layers):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            if i < n_layers:
                for param in layer.parameters():
                    param.requires_grad = False
        print(f'  Couches BERT gelées : {n_layers}/12')

    def forward(self, input_ids, attention_mask, return_attention=False):
        """
        return_attention=True → retourne aussi les poids d'attention
        (utilisé en inférence pour la visualisation XAI)
        """
        # FinBERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # (batch, 128, 768)

        # BiLSTM — retourne toute la séquence
        lstm_out, _ = self.bilstm(hidden_states)   # (batch, 128, 512)

        # Attention — pondère les tokens, masque le padding
        context, attn_weights = self.attention(lstm_out, attention_mask)  # (batch, 512), (batch, 128)

        # Classification
        x = self.drop1(context)
        x = self.relu(self.dense(x))
        x = self.drop2(x)
        logits = self.classifier(x)  # (batch, 3)

        if return_attention:
            return logits, attn_weights
        return logits


# ── Instanciation ─────────────────────────────────────────────────────────────
model = FinBertBiLSTMAttention(BACKBONE, freeze_layers=FREEZE_LAYERS).to(DEVICE)
model.bert.resize_token_embeddings(len(tokenizer))


trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())
print(f'\nParamètres entraînables : {trainable_params:,} / {total_params:,}')

**Optimizer, Scheduler, Loss**

In [ ]:
bert_params = [p for p in model.bert.parameters() if p.requires_grad]
head_params = [
    *model.bilstm.parameters(),
    *model.attention.parameters(),
    *model.dense.parameters(),
    *model.classifier.parameters()
]

optimizer = AdamW([
    {'params': bert_params, 'lr': LR_BERT},
    {'params': head_params, 'lr': LR_HEAD}
], weight_decay=WEIGHT_DECAY)

effective_steps = (len(train_loader) // GRAD_ACCUM) * MAX_EPOCHS
warmup_steps    = int(effective_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=effective_steps
)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f'Steps effectifs : {effective_steps} | Warmup : {warmup_steps}')

**Entrainement**

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, grad_accum):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        logits = model(input_ids, attention_mask)  # return_attention=False par défaut
        loss   = criterion(logits, labels) / grad_accum
        loss.backward()

        total_loss += loss.item() * grad_accum

        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            probs  = torch.softmax(logits, dim=-1)

            total_loss  += loss.item()
            all_preds   += logits.argmax(dim=-1).cpu().tolist()
            all_labels  += labels.cpu().tolist()
            all_probs   += probs.cpu().tolist()

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, acc, f1_macro, all_preds, all_labels, all_probs

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

best_val_f1  = 0.0
patience_cpt = 0
best_epoch   = 0
start_time   = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    train_loss                          = train_epoch(model, train_loader, optimizer, scheduler, criterion, GRAD_ACCUM)
    val_loss, val_acc, val_f1, _, _, _ = evaluate(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    elapsed = time.time() - t0
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'Val Acc: {val_acc:.4f} | '
          f'Val F1: {val_f1:.4f} | '
          f'{elapsed:.0f}s')

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_epoch   = epoch
        patience_cpt = 0
        torch.save(model.state_dict(), MODEL_DIR / 'best_model.pt')
        print(f'  → Meilleur modèle sauvegardé (F1-macro val = {val_f1:.4f})')
    else:
        patience_cpt += 1
        if patience_cpt >= PATIENCE:
            print(f'\nEarly stopping à l\'époque {epoch}')
            break

total_time = (time.time() - start_time) / 60
print(f'\nEntraînement terminé en {total_time:.1f} min')
print(f'Meilleure époque : {best_epoch} | Meilleur val F1-macro : {best_val_f1:.4f}')

**Courbes d'apprentissage**

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_ran, history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val Loss',   marker='o')
axes[0].axvline(x=best_epoch, color='red', linestyle='--', label=f'Best epoch {best_epoch}')
axes[0].set_title('Loss')
axes[0].set_xlabel('Époque')
axes[0].legend()

axes[1].plot(epochs_ran, history['val_f1'], label='03 Val F1-macro', marker='o', color='purple')
axes[1].axhline(y=baseline['f1_macro'],     color='gray',   linestyle='--', label=f'Baseline  ({baseline["f1_macro"]:.4f})')
axes[1].axhline(y=metrics_01['f1_macro'],  color='orange', linestyle='--', label=f'01 FinBERT ({metrics_01["f1_macro"]:.4f})')
axes[1].axhline(y=metrics_02['f1_macro'],  color='blue',   linestyle='--', label=f'02 +BiLSTM ({metrics_02["f1_macro"]:.4f})')
axes[1].set_title('Val F1-macro — comparaison 4 modèles')
axes[1].set_xlabel('Époque')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(RES_DIR / 'learning_curves_03.png', dpi=150)
plt.show()

**Évaluation finale sur le test set**

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR / 'best_model.pt', map_location=DEVICE))

_, test_acc, test_f1, test_preds, test_labels, test_probs = evaluate(
    model, test_loader, criterion
)

f1_per_class = f1_score(test_labels, test_preds, average=None)
roc_auc      = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='macro')

print('=== RÉSULTATS TEST — 03 FinBERT + BiLSTM + Attention ===')
print(f'  Accuracy  : {test_acc:.4f}')
print(f'  F1-macro  : {test_f1:.4f}')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print(f'  F1 négatif : {f1_per_class[0]:.4f}')
print(f'  F1 neutre  : {f1_per_class[1]:.4f}')
print(f'  F1 positif : {f1_per_class[2]:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=LABEL_NAMES))

**Matrice de confusion**

In [ ]:
cm = confusion_matrix(test_labels, test_preds, normalize='true')

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='.2%', cmap='Purples',
    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES
)
plt.title('Matrice de confusion — 03 FinBERT + BiLSTM + Attention (test)')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig(RES_DIR / 'confusion_matrix_03.png', dpi=150)
plt.show()

**Sauvegarde des métriques**

In [ ]:
metrics_03 = {
    'model':                '03_finbert_bilstm_attention',
    'accuracy':             round(test_acc, 4),
    'f1_macro':             round(test_f1, 4),
    'f1_per_class': {
        'negatif':          round(float(f1_per_class[0]), 4),
        'neutre':           round(float(f1_per_class[1]), 4),
        'positif':          round(float(f1_per_class[2]), 4),
    },
    'roc_auc':              round(roc_auc, 4),
    'training_time_minutes': round(total_time, 1),
    'best_epoch':           best_epoch,
    'trainable_params':     trainable_params,
    'val_f1_history':       [round(v, 4) for v in history['val_f1']],
    'frozen_bert_layers':   FREEZE_LAYERS,
    'grad_accum_steps':     GRAD_ACCUM,
}

with open(RES_DIR / 'metrics_03.json', 'w') as f:
    json.dump(metrics_03, f, indent=2)

print('Métriques sauvegardées : results/metrics_03.json')
print(json.dumps(metrics_03, indent=2))